# 05 -- SHAP Explainability

**Purpose:** Visualise SHAP values for M4 (Teacher Shortage Risk) and
M5 (Classroom Infrastructure Quality Trend -- Documented Proxy).
Shows which features drive each district's predicted risk tier.

**Prerequisites:** Run `src/explainability/shap_explainer.py` first.

**Outputs read:** `reports/explainability/*.csv`

**Important:** SHAP values indicate **statistical association**, not causal direction.
M5 predicts an infrastructure-quality risk tier, NOT a learning outcome or dropout rate.

**Global top drivers:**
- M4 Teacher Shortage: enrollment_total, pillar2_infrastructure, pillar2_participation
- M5 Classroom Infra Proxy: pillar2_participation, pillar2_infrastructure, pillar2_digital

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT    = Path.cwd().parent
EXP_DIR = ROOT / 'reports' / 'explainability'

# Check all expected files are present
EXPECTED = [
    'shap_status.csv',
    'm4_shap_values.csv', 'm4_shap_feature_ranking.csv', 'm4_shap_top_driver.csv',
    'm5_shap_values.csv', 'm5_shap_feature_ranking.csv', 'm5_shap_top_driver.csv',
]
for f in EXPECTED:
    p = EXP_DIR / f
    status = 'OK' if p.exists() else 'MISSING -- run src/explainability/shap_explainer.py'
    print(f'  {f:40s} {status}')

In [ ]:
# SHAP run status
status_df = pd.read_csv(EXP_DIR / 'shap_status.csv')
print('SHAP run status:')
print(status_df.to_string(index=False))
assert (status_df['status'] == 'OK').all(), 'SHAP did not complete successfully for all modules'

In [ ]:
# M4 -- Teacher Shortage Risk: global feature importance by mean |SHAP|
m4_rank = pd.read_csv(EXP_DIR / 'm4_shap_feature_ranking.csv')
print('M4 Teacher Shortage Risk -- SHAP feature ranking:')
print(m4_rank.to_string(index=False))

m4_rank_sorted = m4_rank.sort_values('mean_abs_shap')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(m4_rank_sorted['feature'], m4_rank_sorted['mean_abs_shap'], color='#0F766E')
ax.set_xlabel('Mean |SHAP|')
ax.set_title('M4 Teacher Shortage Risk -- SHAP Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# M5 -- Classroom Infrastructure Quality Trend (Documented Proxy)
# NOT a learning-outcome or dropout predictor.
m5_rank = pd.read_csv(EXP_DIR / 'm5_shap_feature_ranking.csv')
print('M5 Classroom Infra Quality Trend (Documented Proxy) -- SHAP feature ranking:')
print(m5_rank.to_string(index=False))

m5_rank_sorted = m5_rank.sort_values('mean_abs_shap')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(m5_rank_sorted['feature'], m5_rank_sorted['mean_abs_shap'], color='#D97706')
ax.set_xlabel('Mean |SHAP|')
ax.set_title('M5 Classroom Infra Quality Trend -- SHAP Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Per-district top SHAP driver for M5 (2024-25 test set, 36 districts)
m5_top = pd.read_csv(EXP_DIR / 'm5_shap_top_driver.csv')
print('M5 top SHAP driver per district (2024-25 test set):')
print(
    m5_top.sort_values('top_shap_abs', ascending=False)
    .to_string(index=False)
)
print(f'\nMost common top driver: {m5_top["top_driver"].mode().iloc[0]}')

In [ ]:
# Per-district top SHAP driver for M4 (2024-25 test set)
m4_top = pd.read_csv(EXP_DIR / 'm4_shap_top_driver.csv')
print('M4 top SHAP driver per district (2024-25 test set):')
print(
    m4_top.sort_values('top_shap_abs', ascending=False)
    .to_string(index=False)
)
print(f'\nMost common top driver: {m4_top["top_driver"].mode().iloc[0]}')